# 1. Import libraries

In [2]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# 2. Load data

In [3]:
# Load data
df_customer = pd.read_csv("../data/raw/olist_customers_dataset.csv")
df_geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
df_order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
df_order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
df_sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
df_products = pd.read_csv("../data/raw/olist_products_dataset.csv")
df_product_category_name_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [4]:
# Create a dictionary to hold all datasets for easy access
datasets = {
    "customers": df_customer,
    "geolocation": df_geolocation,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "orders": df_orders,
    "sellers": df_sellers,
    "products": df_products,
    "category_translation": df_product_category_name_translation,
}

# Function to profile a dataset
def profile_dataset(name, df):
    print(f"Dataset: {name}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Duplicates: {df.duplicated().sum():,}")
    print()
    
    return pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })

# Function to find missing values per column
def missing_values_per_column(df, col_list):
    for col in col_list:
        print(f"Column: {col}")
        print(df[col].isna().sum())

# 3. Data Profiling
## 3.1. Customer data
##### Data Glimpse

In [5]:
df_customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


##### Data Profile

In [6]:
df_customer_profile = profile_dataset("customers", df_customer)
df_customer_profile

Dataset: customers
Rows: 99,441
Columns: 5
Duplicates: 0



,dtype,missing,missing_pct,unique
customer_id,object,0,0.0,99441
customer_unique_id,object,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,object,0,0.0,4119
customer_state,object,0,0.0,27


Customer data seems clean, with no missing values and duplicates. It is generally string data with the zip code being the only numeric data. Olist has almost 100,000 registered customers that have interacted with the platform.

## 3.2. Geolocation
##### Data Glimpse

In [7]:
df_geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


Looking at this data, the coordinate information is grouped by zip codes, which explains the huge amount of rows.

##### Data Profile

In [8]:
df_geolocation_profile = profile_dataset("geolocation", df_geolocation)
df_geolocation_profile

Dataset: geolocation
Rows: 1,000,163
Columns: 5
Duplicates: 261,831



,dtype,missing,missing_pct,unique
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717360
geolocation_lng,float64,0,0.0,717613
geolocation_city,object,0,0.0,8011
geolocation_state,object,0,0.0,27


The geolocation data doesn't seem to have any missing values but it does have quite a lot of duplicated data. This data will need to be cleaned further.

## 3.3. Order Items
##### Data Glimpse

In [9]:
df_order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


##### Data Profile

In [10]:
df_order_items_profile = profile_dataset("order_items", df_order_items)
df_order_items_profile

Dataset: order_items
Rows: 112,650
Columns: 7
Duplicates: 0



,dtype,missing,missing_pct,unique
order_id,object,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,object,0,0.0,32951
seller_id,object,0,0.0,3095
shipping_limit_date,object,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999


The order items dataset seems clean, with no missing values and duplicates, and correct data types.

## 3.4. Order Payments
### Data Glimpse

In [11]:
df_order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Data Profile

In [12]:
df_order_payments_profile = profile_dataset("order_payments", df_order_payments)
df_order_payments_profile

Dataset: order_payments
Rows: 103,886
Columns: 5
Duplicates: 0



,dtype,missing,missing_pct,unique
order_id,object,0,0.0,99440
payment_sequential,int64,0,0.0,29
payment_type,object,0,0.0,5
payment_installments,int64,0,0.0,24
payment_value,float64,0,0.0,29077


The order payments dataset seems clean, with no missing values and duplicates, and correct data types.

## 3.5. Order Reviews
### Data Glimpse

In [13]:
df_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### Data Profile

In [14]:
df_order_reviews_profile = profile_dataset("order_reviews", df_order_reviews)
df_order_reviews_profile

Dataset: order_reviews
Rows: 99,224
Columns: 7
Duplicates: 0



,dtype,missing,missing_pct,unique
review_id,object,0,0.00,98410
order_id,object,0,0.00,98673
review_score,int64,0,0.00,5
review_comment_title,object,87656,88.34,4527
review_comment_message,object,58247,58.70,36159
review_creation_date,object,0,0.00,636
review_answer_timestamp,object,0,0.00,98248


It seems the reviews dataset contains missing values. However, the missing values are `review_comment_title` and `review_comment_message`, which won't always have a value. So, these missing values are not an actual problem.

## 3.6. Orders
### Data Glimpse

In [15]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Data Profile

In [16]:
df_order_profile = profile_dataset("orders", df_orders)
df_order_profile

Dataset: orders
Rows: 99,441
Columns: 8
Duplicates: 0



,dtype,missing,missing_pct,unique
order_id,object,0,0.00,99441
customer_id,object,0,0.00,99441
order_status,object,0,0.00,8
order_purchase_timestamp,object,0,0.00,98875
order_approved_at,object,160,0.16,90733
order_delivered_carrier_date,object,1783,1.79,81018
order_delivered_customer_date,object,2965,2.98,95664
order_estimated_delivery_date,object,0,0.00,459


It seems there are multiple columns with missing values. We can check each one to determine what they mean. The easiest first step is to see each one's `order_status`.

##### `order_approved_at`.

In [17]:
# Analyze the order status for orders that have not been approved
df_orders["order_status"][df_orders["order_approved_at"].isna()].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

`order_approved_at` has missing values across all three statuses. 141 orders were missing aproval time due to being cancelled and 5 were due to just being created. But there are 14 orders that were already delivered yet were missing approval time. Anomalies. Could be caused by other things, such as internal operations.

##### `order_delivered_carrier_date`

In [18]:
df_orders["order_status"][df_orders["order_delivered_carrier_date"].isna()].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

This is date where orders were handled to the logistics partner. There are more order statuses that has missing values. 

609 orders were `unavailable`, which was unclear on what unavailable means in this context. 550 orders were cancelled, which probably included the 141 cancelled orders in the previous check. 314 `invoiced`, 301 `processing`, 5 `created`, and 2 `approved` would mean the orders havent been shipped (given to logistics) yet. 2 `delivered` was another anomaly, which could be caused by internal operations.

##### `order_delivered_customer_date`

In [19]:
df_orders["order_status"][df_orders["order_delivered_customer_date"].isna()].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

This date is where orders were officially delivered to the customers by the logistics partner. 

The missing values for `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created, and `approved` all make sense as it means these orders are not yet delivered. The only anomaly, again, is the 8 orders that have `delivered` status but don't have `order_delivered_customer_date` value.

## 3.7. Sellers

### Data Glimpse

In [20]:
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### Data Profile

In [21]:
# Sellers dataset
df_sellers_profile = profile_dataset("sellers", df_sellers)
df_sellers_profile

Dataset: sellers
Rows: 3,095
Columns: 4
Duplicates: 0



,dtype,missing,missing_pct,unique
seller_id,object,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,object,0,0.0,611
seller_state,object,0,0.0,23


Based on this result, it seems the seller data is proper and clean, no missing values or duplicates. Data types are correct for each column.

## 3.8. Product
### Data Glimpse

In [22]:
df_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### Data Profile

In [23]:
df_products_profile = profile_dataset("products", df_products)
df_products_profile

Dataset: products
Rows: 32,951
Columns: 9
Duplicates: 0



,dtype,missing,missing_pct,unique
product_id,object,0,0.00,32951
product_category_name,object,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95


Based on this result, we can see that there are 2 different results for missing values. 610 missing values for `product_category_name`, `product_name_lenght`, `product_description_lenght` and `product_photos_qty`, and 2 missing values for the weight and dimension columns. We can check whether these `2` rows are part of the `610` or not.

In [24]:
df_products[df_products["product_weight_g"].isna() == True]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Two products are missing physical dimensions (weight, length, height, width). Inspecting these rows reveales:
- One product is missing physical dimensions **only**. It's metadata information is present.
- The other product is missing all fields but the ID.

This means that the only one of the product is a subset of the 610 metadata-missing products. 

## 3.9. Product Category Translation
### Data Glimpse

In [25]:
df_product_category_name_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


This dataset seems to serve as a lookup table to convert the product category name to their english translations.

### Data Profile

In [26]:
df_category_translation = profile_dataset("category_translation", df_product_category_name_translation)
df_category_translation

Dataset: category_translation
Rows: 71
Columns: 2
Duplicates: 0



,dtype,missing,missing_pct,unique
product_category_name,object,0,0.0,71
product_category_name_english,object,0,0.0,71


This dataset seems to be clean. 71 rows, meaning that Olist has 71 product categories. No missing values or duplicated values. Data types are all correct.

## 4. Summary
### 4.1. Data Quality Findings
**Geolocation** has 261,831 duplicated rows out of approximately 1 million rows. This data will need deduplication. However, since the this project won't include geolocation data (as the customer and seller datasets already has their own location attribute), we won't be doing this.

**Orders** have missing timestamps that follow orders lifecycle logic. Cancelled, processing, and unavailable orders will genuinely lack later-stage timestamps because they never reached those stages. However, some records have `delivered` status but missing some timestamps. These are genuine data quality anomalies.

**Order reviews** have missing `review_comment_title` and `review_comment_message`. However, these are optional fields that customer don't have to fill in and are not data quality issue. They will be used to derive `has_comment` measure in `fact_reviews`.

**Products** have two distinct missing values patterns:
- 610 products are missing all metadata fields (category, name length, description length, and photo count) but have physical dimensions
- 2 products are missing physical dimensions, and one of them also belongs to the 610 group.
- The data source contains spelling typos for `product_name_lenght` and `product_description_lenght` (should be `length`).

**Customers, sellers, order items, order payments, and category translations**
are clean. No missing values, no duplicates, correct data types.

### 4.2. Design Decisions Informed by Profiling
Below are some design decisions made based on the findings:

**Finding 1**: Lifecycle-driven null timestamps.<br>
**Decision**: Only include orders with `status=delivered` with non-null delivery dates for analysis. Unfinished orders will be excluded since they won't answer the business questions.

**Finding 2**: Delivered-but-incomplete records <br>
**Decision**: Create a boolean `is_delivery_complete` column for delivered orders with complete lifecycle timestamps. Exclude ones that return `False`.

**Finding 3**: 610 products missing metadata information <br>
**Decision**: Retain with `Unknown` category.

**Finding 4**: 1 product missing dimension information<br>
**Decision**: Retain but won't be able to contribute to analysis regarding product size.

**Finding 5**: `lenght` typos <br>
**Decision**: Rename to `length`.

##### Design Decision Unrelated to Findings
- **Products** and **category translations** will be combined.
- Additional flags in products data: `has_category_name` and `has_dimensions`. Useful for doing category and product-level analysis.

### 4.3. Next Steps
Findings and decisions from this profiling feed directly into:
- `docs/data-model.md` 
- `02_ETL.ipynb` - data cleaning and transormation listed above.